In [1]:
import numpy as np
import sys

scopepath = '/Users/sergivela/Documents/SCOPE/Program/'
sys.path.append(scopepath)

import Scope
from Scope import *
import matplotlib.pyplot as plt
from rdkit.Chem import Draw
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import Descriptors, rdMolDescriptors
from sklearn.decomposition import PCA
from rdkit import Chem
import mols2grid

import yaml
import dscribe
from dscribe.descriptors import SOAP
from ase import Atoms

In [2]:
import os, sys
sys.path.append(os.path.abspath('..'))
from time import time
import sys
import os

# For Jupyter notebooks: get path relative to the current working directory
sys.path.append(os.path.abspath("../QIRs/qirs"))

In [3]:
from ase.build import molecule as ase_molec
from ase.visualize import view
import matplotlib.pyplot as plt
import numpy as np

from qirs import Molecule, SphericalAverageRHFDict
from qirs.core.descriptors import MODA
from qirs.utils.visualization import plot_moda_evolution

ModuleNotFoundError: No module named 'qirs'

In [8]:
systems = load_binary("finished_systems.npy")

Creation of the xyz files, skipping iodine

In [17]:
import os

def write_decorated_extxyz2(fhandle, labels, coord, energy, refcode, spin_state, charge: int = 0, spin: int = 1, mol=None):
    """
    Writes an extended XYZ entry with 'fragment' decorators to a shared file handle.
    """
    natoms = len(labels)

    # Default to all atoms being fragment = -1
    fragment = [-1] * natoms

    if mol is not None:
        metal_idx = next(i for i, atom in enumerate(mol.atoms) if atom.label == "Fe")
        coord_idxs = mol.atoms[metal_idx].get_coord_sphere_idx()
        for idx in [metal_idx] + coord_idxs:
            fragment[idx] = 1
    else:
        raise ValueError("mol argument is required to determine coordination sphere.")

    # Write to shared file handle
    print(natoms, file=fhandle)
    print(f'Properties=species:S:1:fragment:I:1:pos:R:3 charge={charge} spin={spin} energy_base={energy} refcode = {refcode} state = {spin_state}', file=fhandle)
    for l, frag, pos in zip(labels, fragment, coord):
        print(f'{l:2} {frag:2d} {pos[0]:16.10f} {pos[1]:16.10f} {pos[2]:16.10f}', file=fhandle)


output_file_1 = 'all_systems_decorated_HS.xyz'
# Directory containing the .xyz files

# Open the file once in write mode
with open(output_file_1, 'w') as f:
    for sys in systems:
        # HS STATE
        found_HS, HS_state = find_state(sys.HS_ref_mol, "b3lyp_opt")
        refcode = sys.refcode
        if found_HS:
            mol_HS = sys.HS_ref_mol
            if any(atom.label == 'I' for atom in mol_HS.atoms):
                print('Skip') 
                continue
            energy_HS = HS_state.results["energy"].value 
            write_decorated_extxyz2(
                fhandle=f,
                labels=mol_HS.labels,
                coord=mol_HS.coord,
                energy=energy_HS,
                refcode = refcode, 
                spin_state = 'HS',
                charge=mol_HS.totcharge,
                spin=4,
                mol=mol_HS
            )
            print(f"Appended HS: {sys.refcode}")

print(f"\n All structures HS saved in: {output_file_1}")

output_file_2 = 'all_systems_decorated_LS.xyz'
# Open the file once in write mode
with open(output_file_2, 'w') as f:
    for sys in systems:
        # LS STATE
        found_LS, LS_state = find_state(sys.LS_ref_mol, "b3lyp_opt")
        refcode = sys.refcode
        if found_LS:
            mol_LS = sys.LS_ref_mol
            if any(atom.label == 'I' for atom in mol_LS.atoms):
                print('Skip')
                continue
            energy_LS = LS_state.results["energy"].value 
            write_decorated_extxyz2(
                fhandle=f,
                labels=mol_LS.labels,
                coord=mol_LS.coord,
                energy=energy_LS,
                refcode = refcode, 
                spin_state = 'LS',
                charge=mol_LS.totcharge,
                spin=0,
                mol=mol_LS
            )
            print(f"Appended LS: {sys.refcode}")

print(f"\n All structures LS saved in: {output_file_2}")



Appended HS: ABITEM
Appended HS: ACAHUJ02
Appended HS: ADAQUU
Appended HS: ADAREF
Appended HS: ADEJOK
Appended HS: ADEQIL
Appended HS: ADEQOR
Appended HS: AFANEB02
Appended HS: AFOYUS02
Appended HS: AFUHOB01
Appended HS: AGIZEX
Appended HS: AHINUD
Appended HS: AJAXEP
Appended HS: AKEMUY
Appended HS: AKENAF
Appended HS: AKEQEM
Appended HS: ALIKUC
Appended HS: ALILEN
Appended HS: ALUBOB
Appended HS: AMIHUA
Appended HS: APAFEH03
Appended HS: APINUJ
Appended HS: ARADOO
Appended HS: AROZIS
Appended HS: ATASEU
Appended HS: ATAVAU
Appended HS: ATAVEY01
Appended HS: ATAXAW
Appended HS: ATAXEB
Appended HS: ATOZUG03
Appended HS: ATUBIC
Appended HS: ATUBOI01
Appended HS: AWUKEK01
Appended HS: AXAKIT01
Appended HS: BADLIF
Appended HS: BADWAG01
Appended HS: BAKGUR01
Appended HS: BALBIB
Appended HS: BAXFIS03
Appended HS: BAXJUI
Appended HS: BAXXAB05
Appended HS: BAYKUJ
Appended HS: BAYZEH
Appended HS: BEFNIM
Appended HS: BEFNOS
Appended HS: BEGXOB
Appended HS: BEHCEY
Appended HS: BEPZIF
Appended HS:

Use xyz files to create moda for LS and HS (took about a day and a half in my computer) 

In [ ]:
import pickle
import sys
import os
import shutil
import numpy as np

import Scope
from Scope import *
from ase.io import read
from qirs import Molecule  
from qirs.core.descriptors import MODA
from qirs.utils.io import make_results_df

# RHFDict initialization (reuse for all molecules)
RHFDict = SphericalAverageRHFDict('def2svp')

# Load all systems
moda_LS, energies_LS = MODA.get_from_file("all_systems_decorated_LS.xyz", basis="def2svp", verbose=True)  ## Create LS_molecules trajectory file 
moda_HS, energies_HS = MODA.get_from_file("all_systems_decorated_HS.xyz", basis="def2svp", verbose=True)  ## Create HS_molecules trajectory file
df_LS = make_results_df(moda_LS, energies_LS)
df_HS = make_results_df(moda_HS, energies_HS)

Separate inner/outer

In [ ]:
def split_moda(moda):
    inner, outer = {}, {}
    for pair, val in moda.items():
        atom1 = pair.split("-")[0]
        atom2 = pair.split("-")[1]
        if 'N_1' in atom2 and 'Fe_1' in atom1 or 'N_1' in atom1 and 'Fe_1' in atom2:
            inner[pair] = val
        elif "Fe_" in atom1 and "Fe_" in atom2: ## Since there is only one Fe atom, Fe-Fe interactions will be 0
            pass 
        elif "N_" in atom1 and "N_" in atom2:   ## I think interactions between N_1 atoms go to the outer part. I think this is what we did with scope, but please verify
            outer[pair] = val
        else:
            outer[pair] = val
    return inner, outer

In [ ]:
# Split MODA contributions
rho_inner_HS, rho_outer_HS = split_moda(moda_HS)
rho_inner_LS, rho_outer_LS = split_moda(moda_LS)

Save as variables for the systems (Skipping iodine)

In [ ]:
j = 0  # index for rho and energy lists
for i, sys in enumerate(systems):
    mol_LS = sys.LS_ref_mol
    if any(atom.label == 'I' for atom in mol_LS.atoms):
        print(f"Skip system {i} (contains iodine)")
        continue

    # Safely assign the j-th values (which correspond only to non-I systems)
    sys.HS_ref_mol.rho_inner_HS = {k: v[j] for k, v in rho_inner_HS.items()}
    sys.HS_ref_mol.rho_outer_HS = {k: v[j] for k, v in rho_outer_HS.items()}
    sys.HS_ref_mol.H_HS = energies_HS[j]

    sys.LS_ref_mol.rho_inner_LS = {k: v[j] for k, v in rho_inner_LS.items()}
    sys.LS_ref_mol.rho_outer_LS = {k: v[j] for k, v in rho_outer_LS.items()}
    sys.LS_ref_mol.H_LS = energies_LS[j]

    j += 1  # Only increment if system was processed


In [ ]:
save_binary(systems, "systems_with_moda_separated.npy")

Use the saved file 

In [5]:
systems_with_moda = load_binary("systems_with_moda_separated.npy")

In [7]:
# Initialize empty lists
rho_inner_HS_all = []
rho_outer_HS_all = []
rho_inner_LS_all = []
rho_outer_LS_all = []
energies_HS_all = []
energies_LS_all = []

for sys in systems_with_moda:
    mol_LS = sys.LS_ref_mol
    if hasattr(mol_LS, "rho_inner_LS"):  # only collect if data was assigned
        rho_inner_HS_all.append(sys.HS_ref_mol.rho_inner_HS)
        rho_outer_HS_all.append(sys.HS_ref_mol.rho_outer_HS)
        energies_HS_all.append(sys.HS_ref_mol.H_HS)

        rho_inner_LS_all.append(sys.LS_ref_mol.rho_inner_LS)
        rho_outer_LS_all.append(sys.LS_ref_mol.rho_outer_LS)
        energies_LS_all.append(sys.LS_ref_mol.H_LS)

Substraction HS/LS (not sure it's well done)

In [8]:
import numpy as np

# Turn dicts into flat arrays
inner_total = [
    np.array(list(rho_inner_HS_all[i].values())) - np.array(list(rho_inner_LS_all[i].values()))
    for i in range(len(rho_inner_HS_all))
]

outer_total = [
    np.array(list(rho_outer_HS_all[i].values())) - np.array(list(rho_outer_LS_all[i].values()))
    for i in range(len(rho_outer_HS_all))
]

# Energies
dH = np.array(energies_HS_all) - np.array(energies_LS_all)

# Optional: check shapes
print(np.array(inner_total).shape)  # should be (n_samples, n_features)
print(np.array(outer_total).shape)
print(dH.shape)


(722, 66)
(722, 6423)
(722,)


Kernels

In [9]:
from scipy.spatial.distance import cdist

# RBF kernel
def k(rho, l):
    return np.exp(-l * cdist(rho, rho, 'sqeuclidean'))

# Combined kernel for inner and outer descriptors
def total_kernel(inner_list, outer_list, l_inner, l_outer):
    return np.prod(
        [k(rho, l) for rho, l in zip([inner_list, outer_list], [l_inner, l_outer])],
        axis=0
    )


In [13]:
K = total_kernel(inner_total, outer_total, 1, 0.1)
print("Inner norms:", [np.linalg.norm(v) for v in inner_total])
print("Outer norms:", [np.linalg.norm(v) for v in outer_total])

Inner norms: [0.23159232879404565, 0.2097133624120287, 0.29814448802860893, 0.33426936713430216, 0.0022435774985197746, 0.0005649101847185787, 0.00044350985525172493, 0.26800776119528646, 0.23347091862043523, 0.0008216468951051795, 0.0027361556542618492, 0.03300798695897657, 0.0027728205457982946, 0.06913970278590371, 0.21185495871826718, 0.0037607436343352115, 0.007843515918292129, 0.003314204042088913, 0.005375628099205235, 0.005701233534995259, 0.006427186284221299, 0.0043204082240321465, 0.0019070648269181845, 0.007790413932469484, 0.0014083522702100953, 0.0015831513475958886, 0.2460937911617427, 0.005431250140731202, 0.001876674123533156, 0.006841105842739424, 0.0068440340691231245, 0.006866932819693612, 0.003638341639654993, 0.003850538262476455, 0.007683414625941666, 0.0054525493382011, 0.2726702966207979, 0.0013964133250451412, 0.2616090198621443, 0.0013716912737194564, 0.1777347901975592, 0.0020867120481186004, 0.23115273868846836, 0.0020812532443273994, 0.0020900997399892853,

In [17]:
print(f"ΔH range: {min(dH):.3f} to {max(dH):.3f}")
print(f"ΔH std dev: {np.std(dH):.4f}")

ΔH range: -0.050 to 0.113
ΔH std dev: 0.0167


R^2 is way too small, so there is a big issue here in the sense that, this below, is wrong. But the error could be due to the substraction of HS/LS

In [12]:
import numpy as np
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import mean_absolute_error, r2_score
from collections import Counter

# Hyperparameters
l_inner_list = np.logspace(-5, 5, 5)  
ratios = [0.001, 0.01, 0.1, 1.0]
alpha = 1e-5
n_repeats = 5

# Data setup
n = len(dH)
split = int(0.80 * n)
all_indices = np.arange(n)

results = []
best_score = float('inf')
best_model_info_rest = {}
best_true = None
best_pred = None

for l_inner in l_inner_list:
    for ratio in ratios:
        l_outer = l_inner * ratio
        print(f"\nTesting l_inner={l_inner:.6e}, l_outer={l_outer:.6e}")
        
        K = total_kernel(inner_total, outer_total, l_inner, l_outer)

        maes = []
        r2s = []
        all_outliers = []
        split_outliers = []
        outlier_counter = Counter()

        all_preds = []
        all_trues = []

        for repeat in range(n_repeats):
            shuffled_indices = np.copy(all_indices)
            np.random.seed(repeat)
            np.random.shuffle(shuffled_indices)

            train_idx = shuffled_indices[:split]
            test_idx = shuffled_indices[split:]

            K_train = K[train_idx][:, train_idx]
            K_test = K[test_idx][:, train_idx]
            dH_train = dH[train_idx]
            dH_test = dH[test_idx]

            model = KernelRidge(kernel="precomputed", alpha=alpha)
            model.fit(K_train, dH_train)
            dH_pred = model.predict(K_test)
            
            mae = mean_absolute_error(dH_test, dH_pred)
            r2 = r2_score(dH_test, dH_pred)
            maes.append(mae)
            r2s.append(r2)

            residuals = np.abs(dH_pred - dH_test)
            threshold = 3 * mae
            outlier_indices = np.where(residuals > threshold)[0]
            absolute_outlier_indices = test_idx[outlier_indices]

            all_preds.append(dH_pred)
            all_trues.append(dH_test)

        mean_mae = np.mean(maes)
        std_mae = np.std(maes)
        mean_r2 = np.mean(r2s)
        std_r2 = np.std(r2s)

        model_info_rest = {
            "l_inner": l_inner,
            "l_outer": l_outer,
            "ratio": ratio,
            "MAE": mean_mae,
            "MAE_std": std_mae,
            "R2": mean_r2,
            "R2_std": std_r2,
        }

        results.append(model_info_rest)

        print(f"→ MAE: {mean_mae:.4f} ± {std_mae:.4f}, "
              f"R²: {mean_r2:.4f} ± {std_r2:.4f}, (over {n_repeats} splits)")

        if mean_mae < best_score:
            best_score = mean_mae
            best_model_info_rest = model_info_rest
            best_true = np.concatenate(all_trues)
            best_pred = np.concatenate(all_preds)



Testing l_inner=1.000000e-05, l_outer=1.000000e-08
→ MAE: 0.0100 ± 0.0009, R²: 0.0111 ± 0.0258, (over 5 splits)

Testing l_inner=1.000000e-05, l_outer=1.000000e-07
→ MAE: 0.0100 ± 0.0009, R²: 0.0166 ± 0.0266, (over 5 splits)

Testing l_inner=1.000000e-05, l_outer=1.000000e-06
→ MAE: 0.0101 ± 0.0010, R²: 0.0209 ± 0.0291, (over 5 splits)

Testing l_inner=1.000000e-05, l_outer=1.000000e-05
→ MAE: 0.0101 ± 0.0010, R²: 0.0268 ± 0.0374, (over 5 splits)

Testing l_inner=3.162278e-03, l_outer=3.162278e-06
→ MAE: 0.0099 ± 0.0009, R²: 0.0292 ± 0.0385, (over 5 splits)

Testing l_inner=3.162278e-03, l_outer=3.162278e-05
→ MAE: 0.0099 ± 0.0010, R²: 0.0339 ± 0.0678, (over 5 splits)

Testing l_inner=3.162278e-03, l_outer=3.162278e-04
→ MAE: 0.0100 ± 0.0011, R²: 0.0088 ± 0.1181, (over 5 splits)

Testing l_inner=3.162278e-03, l_outer=3.162278e-03
→ MAE: 0.0103 ± 0.0010, R²: -0.0048 ± 0.1212, (over 5 splits)

Testing l_inner=1.000000e+00, l_outer=1.000000e-03
→ MAE: 0.0104 ± 0.0017, R²: -0.1028 ± 0.340

In [16]:
# Final best result
print("\nBest model (avg over splits):")
for i, v in best_model_info_rest.items():
    if i == "outliers":
        print(f"{i}: {v}")
    elif isinstance(v, float):
        print(f"{i}: {v:.7f}")
    else:
        print(f"{i}: {v}")


Best model (avg over splits):
l_inner: 316.2277660
l_outer: 316.2277660
ratio: 1.0000000
MAE: 0.0098288
MAE_std: 0.0009567
R2: 0.0511843
R2_std: 0.0656255
